In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (classification_report, roc_auc_score, 
                             average_precision_score, confusion_matrix)
from sklearn.preprocessing import label_binarize

df = pd.read_csv('../data/processed/dataset1_processed.csv')
X = df.drop(columns=['diagnosis'])
y = df['diagnosis'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':           SVC(kernel='rbf', probability=True, random_state=42),
    'MLP':           MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    report = classification_report(y_test, y_pred, output_dict=True)
    auc_roc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
    y_bin = label_binarize(y_test, classes=[1, 2, 3])
    auc_pr = average_precision_score(y_bin, y_proba, average='macro')
    
    results.append({
        'Model':         name,
        'Accuracy': round(report['accuracy'], 3),
        'Precision (macro)': round(report['macro avg']['precision'], 3),
        'Recall (macro)': round(report['macro avg']['recall'], 3),
        'F1 (macro)': round(report['macro avg']['f1-score'], 3),
        'AUC-ROC': round(auc_roc, 3),
        'AUC-PR': round(auc_pr, 3),
    })
    
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Healthy','Benign','Cancer']))
    print(f"  AUC-ROC : {auc_roc:.3f}")
    print(f"  AUC-PR  : {auc_pr:.3f}")

summary = pd.DataFrame(results).set_index('Model')
print("\n── Summary ──────────────────────────────────────────")
print(summary.to_string())


  Random Forest
              precision    recall  f1-score   support

     Healthy       0.95      1.00      0.97        37
      Benign       0.95      0.93      0.94        41
      Cancer       0.97      0.95      0.96        40

    accuracy                           0.96       118
   macro avg       0.96      0.96      0.96       118
weighted avg       0.96      0.96      0.96       118

  AUC-ROC : 0.991
  AUC-PR  : 0.985

  SVM
              precision    recall  f1-score   support

     Healthy       0.77      0.89      0.82        37
      Benign       0.79      0.80      0.80        41
      Cancer       0.91      0.75      0.82        40

    accuracy                           0.81       118
   macro avg       0.82      0.82      0.81       118
weighted avg       0.82      0.81      0.81       118

  AUC-ROC : 0.907
  AUC-PR  : 0.806

  MLP
              precision    recall  f1-score   support

     Healthy       0.89      0.89      0.89        37
      Benign       0.74   

In [3]:
import joblib

rf_model = models['Random Forest']
joblib.dump(rf_model,'../models/ds1/ds1_random_forest.pkl')

svm_model = models['SVM']
joblib.dump(svm_model,'../models/ds1/ds1_svm.pkl')

mlp_model = models['MLP']
joblib.dump(mlp_model,'../models/ds1/ds1_mlp.pkl')

#save the split and features
np.save('../models/ds1/X_train.npy', X_train)
np.save('../models/ds1/X_test.npy', X_test)
np.save('../models/ds1/y_test.npy', y_test)
pd.Series(X.columns.tolist()).to_csv('../models/ds1/feature_names.csv', index=False)